## 1. Imports and path

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import warnings
from pathlib import Path
from itertools import product           #used to produce the combinations
from scipy.stats import wilcoxon, binomtest

warnings.filterwarnings('ignore')
np.random.seed(36)

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 60)
pd.set_option('display.float_format', '{:.5f}'.format)

PROJECT_ROOT=Path.cwd().parent
PHASE6_OUTPUT=PROJECT_ROOT / 'data' / 'processed' / 'phase6_models'
PHASE7_OUTPUT=PROJECT_ROOT / 'data' / 'processed' / 'phase7_stats'
PHASE7_OUTPUT.mkdir(parents=True, exist_ok=True)

BASELINE_MODEL='pointwise'
BASELINE_PIPELINE='raw'
BASELINE_KEY=f'{BASELINE_MODEL}_{BASELINE_PIPELINE}'

MODELS=['pointwise', 'pairwise', 'lightgbm']
PIPELINES=['raw', 'global', 'per_query']
DATASETS =['2007', '2008']

COMPARE_CONFIGS=[
    f'{m}_{p}'
    for m, p in product(MODELS, PIPELINES)      #generates all combinations
    if f'{m}_{p}' != BASELINE_KEY
]

FDR_ALPHA=0.05                  #False Discovery Rate
N_BOOTSTRAP=2000                #how many times we "resample" queries to estimate uncertainty
EFFECT_SMALL=0.10               #minimum practical effect size threshold for NDCG@5 and P@5
FAILURE_EFFECT_SMALL=0.01       #don't claim a meaningful change unless the absolute change is at least 1%

print('='*80)
print('Phase 7 = STATISTICAL VALIDATION ONLY')
print('='*80)
print(f'Phase 7 output  : {PHASE7_OUTPUT}')
print(f'Baseline config : {BASELINE_KEY}')
print(f'Configs to test : {len(COMPARE_CONFIGS)}')
print(f'Datasets        : {DATASETS}')
print(f'FDR alpha       : {FDR_ALPHA}')
print(f'Bootstrap reps  : {N_BOOTSTRAP}')
print(f'Cliff delta thresh  : {EFFECT_SMALL}')
print(f'Risk diff thresh: {FAILURE_EFFECT_SMALL}')
print('='*80)

Phase 7 = STATISTICAL VALIDATION ONLY
Phase 7 output  : b:\Arlington\Arlington\4th_sem\Capstone\Capstone_coding\data\processed\phase7_stats
Baseline config : pointwise_raw
Configs to test : 8
Datasets        : ['2007', '2008']
FDR alpha       : 0.05
Bootstrap reps  : 2000
Cliff delta thresh  : 0.1
Risk diff thresh: 0.01


## 2. Utility functions

In [ ]:
# 1. failure-flag parser 

def make_fail_flag(series: pd.Series)->pd.Series:
    #Converting Failure@5_primary to clean 0/1 int. No silent misclassification.
    if pd.api.types.is_bool_dtype(series):
        return series.astype(int)
    if pd.api.types.is_numeric_dtype(series):
        return series.fillna(0).astype(float).round().astype(int).clip(0, 1)
    _T = {'true', '1', 'yes', '1.0'}
    _F = {'false', '0', 'no', '0.0', ''}
    def _p(v):
        if pd.isna(v):
            return 0
        if isinstance(v, bool):
            return int(v)
        if isinstance(v, (int, float)):
            return 0 if np.isnan(float(v)) else int(round(float(v)))
        if isinstance(v, str):
            vl = v.strip().lower()
            if vl in _T: return 1
            if vl in _F: return 0
        return 0
    return series.map(_p).astype(int)

#self-test
_ts = pd.Series([True, False, 'True', 'false', '1', '0', 1.0, 0.0])
assert list(make_fail_flag(_ts))==[1,0,1,0,1,0,1,0]
del _ts
print('make_fail_flag')


# 2.  Paired merge by qid intersection 
def paired_merge(df_a: pd.DataFrame, df_b: pd.DataFrame,
                 metric: str, evaluable_only: bool = True):
    """
    Return (vec_a, vec_b, qids) aligned by qid intersection.
    If evaluable_only, restrict to qids with num_relevant_1 > 0 in BOTH.
    Raises RuntimeError if required columns are missing.
    """
    required=['qid', metric]
    if evaluable_only:
        required.append('num_relevant_1')
    for col in required:
        if col not in df_a.columns:
            raise RuntimeError(f'Missing column "{col}" in df_a')
        if col not in df_b.columns:
            raise RuntimeError(f'Missing column "{col}" in df_b')

    if evaluable_only:
        a=df_a[df_a['num_relevant_1'] > 0][['qid', metric]].copy()
        b=df_b[df_b['num_relevant_1'] > 0][['qid', metric]].copy()
    else:
        a=df_a[['qid', metric]].copy()
        b=df_b[['qid', metric]].copy()
    merged=a.merge(b, on='qid', suffixes=('_a', '_b'))
    va=merged[f'{metric}_a'].values
    vb=merged[f'{metric}_b'].values
    assert len(va)==len(vb), 'Paired vectors length mismatch after merge'
    return va, vb, merged['qid'].values

print('paired_merge')



# 3.  Wilcoxon wrapper 

def wilcoxon_test(va: np.ndarray, vb: np.ndarray):
    """
    Wilcoxon signed-rank test on paired samples (vb = config, va = baseline).
    Returns (stat, pval, note).
    Guards:
    - If all differences are zero -> p=1.0
    - If zero differences remain after filtering -> p=1.0
    """
    diff=(vb-va).astype(float)

    if np.all(diff==0):
        return 0.0, 1.0, 'no differences observed'

    #filtering zeros, then testing on nonzero
    nonzero=diff[diff != 0]
    if len(nonzero)==0:
        return 0.0, 1.0, 'all differences became zero after filtering'

    try:
        #running wilcoxon on nonzero
        stat, pval=wilcoxon(nonzero, zero_method='wilcox', alternative='two-sided')
        return float(stat), float(pval), ''
    except Exception as e:
        return 0.0, 1.0, f'wilcoxon error: {e}'

print('wilcoxon_test')



# 4.  McNemar's test (exact binomial)

def mcnemar_test(fa: np.ndarray, fb: np.ndarray):
    """
    Exact binomial McNemar on paired binary vectors.
    fa=baseline flags, fb=config flags (both 0/1 int arrays).
    Returns (b01, b10, pval, note).
    b01=baseline=0 config=1  (new failures)
    b10=baseline=1 config=0  (resolved failures)
    """
    b01=int(((fa==0) & (fb==1)).sum())
    b10=int(((fa==1) & (fb==0)).sum())
    discordant=b01+b10
    if discordant==0:
        return b01, b10, 1.0, 'no discordant pairs'
    result=binomtest(min(b01, b10), discordant, 0.5, alternative='two-sided')
    return b01, b10, float(result.pvalue), ''

print('mcnemar_test')



# 5.  Cliff's delta (from paired differences)

def cliffs_delta_paired(va: np.ndarray, vb: np.ndarray) -> float:
    """
    Cliff's delta estimated from paired differences d=vb-va.
    =(number of d>0 - number of d<0) / n
    Range: [-1, 1].  Positive -> config tends to be higher.
    """
    d=vb-va
    n=len(d)
    if n==0:
        return 0.0
    return float((np.sum(d > 0)-np.sum(d < 0)) / n)

print('cliffs_delta_paired')



# 6.  Bootstrap CI for median difference 

def bootstrap_ci_median_diff(va: np.ndarray, vb: np.ndarray,
                              n_boot: int=N_BOOTSTRAP,
                              ci: float=0.95) -> tuple:
    """
    95% bootstrap CI for median(vb-va) using percentile method.
    Returns (lower, upper).
    """
    diff=(vb - va).astype(float)
    n=len(diff)
    if n==0:
        return 0.0, 0.0
    medians=np.array([
        np.median(diff[np.random.randint(0, n, n)])
        for _ in range(n_boot)
    ], dtype=float)
    alpha=(1 - ci) / 2
    return (
        float(np.percentile(medians, 100 * alpha)),
        float(np.percentile(medians, 100 * (1 - alpha)))
    )

print('bootstrap_ci_median_diff')



# 7.  Bootstrap CI for risk difference 

def bootstrap_ci_risk_diff(fa: np.ndarray, fb: np.ndarray,
                            n_boot: int = N_BOOTSTRAP,
                            ci: float = 0.95) -> tuple:
    """
    95% bootstrap CI for risk_diff=mean(fb - fa).
    Proper paired resampling.
    """
    diff=fb-fa
    n=len(diff)
    if n==0:
        return 0.0, 0.0
    boot_stats=[]
    for _ in range(n_boot):
        idx=np.random.randint(0, n, n)
        boot_stats.append(diff[idx].mean())
    alpha=(1-ci)/2
    return (
        float(np.percentile(boot_stats, 100 * alpha)),
        float(np.percentile(boot_stats, 100 * (1 - alpha)))
    )

print('bootstrap_ci_risk_diff')



# 8.  Benjamini-Hochberg FDR correction 

def bh_fdr(pvals: list, alpha: float = FDR_ALPHA) -> np.ndarray:
    """
    Benjamini-Hochberg FDR 
    Returns array of adjusted q-values (same order as input).
    """
    pvals=np.asarray(pvals, dtype=float)
    n=len(pvals)
    if n==0:
        return np.array([])
    order=np.argsort(pvals)
    ranks=np.arange(1, n + 1)
    adj=np.minimum(pvals[order] * n / ranks, 1.0)
    adj=np.minimum.accumulate(adj[::-1])[::-1]
    q=np.empty(n)
    q[order]=adj
    return q

#quick test
_pv=[0.001, 0.002, 0.5, 0.04, 0.8]
_q=bh_fdr(_pv)
assert _q[0]<_q[2], 'BH ordering broken'
del _pv, _q
print('bh_fdr')

print('\nAll utility functions defined')

make_fail_flag
paired_merge
wilcoxon_test
mcnemar_test
cliffs_delta_paired
bootstrap_ci_median_diff
bootstrap_ci_risk_diff
bh_fdr

All utility functions defined


Wilcoxon -> Is improvement statistically detectable?

Cliff’s delta -> Is improvement strong and consistent?

Bootstrap CI -> Is improvement stable?

Risk difference threshold -> Is improvement practically meaningful?

## 3. Loading Phase 6 artifacts

In [7]:
print('='*80)
print('LOADING PHASE 6 QUERY METRICS')
print('='*80)

qm:dict[str, pd.DataFrame]={}

for model, pipeline, dataset in product(MODELS, PIPELINES, DATASETS):
    key =f'{model}_{pipeline}_{dataset}'
    fpath=PHASE6_OUTPUT / f'{key}_query_metrics.csv'
    if not fpath.exists():
        raise RuntimeError(f'Missing required file: {fpath}')
    df=pd.read_csv(fpath)
    df['_fail_flag']=make_fail_flag(df['Failure@5_primary'])
    qm[key]=df
    print(f'{key:40s}  ({len(df)} queries)')

expected=len(MODELS)*len(PIPELINES)*len(DATASETS)
if len(qm)!=expected:
    raise RuntimeError(f'Expected {expected} files, loaded {len(qm)}')

print(f'\nLoaded {len(qm)} / {expected} files')
print('='*80)

LOADING PHASE 6 QUERY METRICS
pointwise_raw_2007                        (336 queries)
pointwise_raw_2008                        (156 queries)
pointwise_global_2007                     (336 queries)
pointwise_global_2008                     (156 queries)
pointwise_per_query_2007                  (336 queries)
pointwise_per_query_2008                  (156 queries)
pairwise_raw_2007                         (336 queries)
pairwise_raw_2008                         (156 queries)
pairwise_global_2007                      (336 queries)
pairwise_global_2008                      (156 queries)
pairwise_per_query_2007                   (336 queries)
pairwise_per_query_2008                   (156 queries)
lightgbm_raw_2007                         (336 queries)
lightgbm_raw_2008                         (156 queries)
lightgbm_global_2007                      (336 queries)
lightgbm_global_2008                      (156 queries)
lightgbm_per_query_2007                   (336 queries)
lightgbm_per_query